# Chapter 07 · Software Development & Data Analysis Agents
## LLM을 활용한 소프트웨어 개발 & 데이터 분석 에이전트
### *Generative AI with LangChain — 2nd Edition*

7장의 내용을 **슬라이드 흐름(Part 1 → 2 → 3)에 따라 개념 설명과 실행 코드를 함께** 정리한 통합 노트북입니다.
각 절은 *핵심 개념(markdown)* → *실행 가능한 코드(code)* 순서로 구성됩니다.

---
### 목차

| Part | 주제 | 절 |
|------|------|----|
| **0** | 환경 설정 | API 키 로드 |
| **1** | LLMs in Software Development | 1.1 Vibe Coding · 1.2 생산성 · 1.3 진화 · 1.4 활용영역 · 1.5 벤치마크 · 1.6 SE접근 · 1.7 보안검증 · 1.8 검증프레임워크 · 1.9 LangChain 통합 |
| **2** | Writing Code with LLMs | 2.1 Gemini · 2.2 Hugging Face · 2.3 Claude · 2.4 에이전트 · 2.5 문서 RAG · 2.6 저장소 RAG |
| **3** | LLM Agents for Data Science | 3.1 보완관계 · 3.2 PyTorch 에이전트 · 3.3 pandas 에이전트 · 3.4 대용량 전략 · 3.5 능력 정리 |

> ⚠️ 코드 셀 대부분은 API 키(OpenAI / Anthropic / Google / HuggingFace)가 필요합니다. **Part 0**을 먼저 실행하세요.


---
# Part 0 · 환경 설정

상위 폴더의 `.env`에서 API 키를 로드합니다. Jupyter에는 `__file__`이 없으므로
**현재 작업 폴더(`Path.cwd()`)** 기준으로 상위 폴더의 `.env`를 찾습니다.

`.env` 예시:
```
OPENAI_API_KEY=sk-...
ANTHROPIC_API_KEY=sk-ant-...
GOOGLE_API_KEY=...
HUGGINGFACEHUB_API_TOKEN=hf_...
```


In [ ]:
# %pip install -q python-dotenv
from pathlib import Path
from dotenv import load_dotenv
import os

for candidate in [Path.cwd().parent / ".env", Path.cwd() / ".env"]:
    if candidate.exists():
        load_dotenv(candidate)
        print("loaded:", candidate)
        break
else:
    print("⚠️ .env 파일을 찾지 못했습니다. 경로를 확인하세요.")

for k in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GOOGLE_API_KEY", "HUGGINGFACEHUB_API_TOKEN"]:
    print(f"{k}: {'OK' if os.getenv(k) else '미설정'}")

필요 패키지 (한 번만 설치):

In [ ]:
# %pip install -qU langchain langchain-openai langchain-anthropic langchain-google-genai \
#     langchain-huggingface langchain-community langchain-experimental langchain-chroma \
#     transformers torch pandas matplotlib datasets evaluate chromadb scikit-learn

---
# Part 1 · LLMs in Software Development
> 자연어가 코드를 만들고, 코드가 시스템을 움직인다.


## 1.1 자연어와 프로그래밍 — "Vibe Coding"

**Vibe Coding**: Andrej Karpathy가 2025년 초 대중화. 문법·자료구조를 직접 다루지 않고 원하는 동작을 **자연어로 묘사**하면 모델이 코드를 조립한다. (*"디버그하지 않고, 다시 vibe 한다."*)
대표 도구: Cursor · Windsurf · OpenHands · Amazon Q Developer

**핵심**
- 자연어는 대체가 아닌 **'고수준 인터페이스'** — 전통 언어와 공존
- 프로토타이핑은 빠르게, 정밀 구현은 코드로 / 속도 vs 품질·보안 균형이 핵심
- IDC: 2028년 신규 디지털 솔루션의 **70%**가 자연어로 작성될 전망


## 1.2 현실 점검 — 생산성

| 구분 | 수치 | 비고 |
|------|------|------|
| 벤더 발표 | **55%** | GitHub Copilot 통제 실험 (한정된 과업) |
| 독립 연구 | **4–22%** | BlueOptima 218,000명 분석 등 실 환경 |

독립 연구: 88%가 배포 전 재작업 필요 · AI 보조 시 더 높은 버그율 · SWE-Lancer 최고 모델도 26.2%만 완료. ➡️ **검증이 필수**.


In [ ]:
# 벤더 주장 vs 독립 연구 — 간단 비교 시각화
import matplotlib.pyplot as plt

labels = ["Vendor (Copilot)", "Independent (low)", "Independent (high)"]
values = [55, 4, 22]
plt.figure(figsize=(6, 3))
plt.bar(labels, values, color=["#4C8BF5", "#F5A623", "#F5A623"])
plt.ylabel("생산성 향상 (%)")
plt.title("AI 코딩 도구 생산성: 주장 vs 측정")
for i, v in enumerate(values):
    plt.text(i, v + 1, f"{v}%", ha="center")
plt.tight_layout(); plt.show()

## 1.3 코드 LLM의 진화 — 세 단계

| 단계 | 시기 | 특징 |
|------|------|------|
| Foundation | 2021–초2022 | 최초 실용 코드 생성 모델, 가능성 증명 |
| Expansion | 말2022–초2023 | 추론력·맥락 약진, 다중 언어 |
| Diversification | 2023중–2024 | 고도 상업 모델 + 강력한 오픈소스 병행 |

상업: Codex → Copilot → GPT-4 / Claude 3.5 Sonnet → Gemini 2.5 Pro / 오픈소스: StarCoder → Code Llama → CodeGemma → Codestral


## 1.4 코드 LLM 활용 영역 — 6가지

1. **Code Generation** 2. **Test Generation** 3. **Code Documentation**
4. **Editing/Refactor** 5. **Code Translation** 6. **Debug & Auto-repair** (SWE-agent · AutoCodeRover · RepoUnderstander)


## 1.5 벤치마크 — 어떤 모델을 고를까

| Benchmark | 특징 | 시사점 |
|-----------|------|--------|
| HumanEval | 164개 Python 함수, pass@1 | o1 92.4% / Claude 3 Opus 84.9% |
| MBPP | ~974개 입문 과제 | 맥락 부족, 실서비스와 거리 |
| ClassEval | 클래스 수준 | 함수 대비 15–30% 하락 |
| SWE-bench | 실제 GitHub 버그 수정 | 최상위도 40–65% |

> 벤치마크는 정답이 아니다 — 다중 파일·상태·도구 호출 맥락은 측정되지 않음.

아래는 pass@k를 직접 계산하는 실습 코드다.


In [ ]:
# pass@k 평가 — HumanEval식 code_eval
# %pip install -q evaluate datasets
import os
os.environ["HF_ALLOW_CODE_EVAL"] = "1"
from evaluate import load

code_eval_metric = load("code_eval")
test_cases = ["assert add(2, 3) == 5"]
candidates = [["def add(a, b): return a*b", "def add(a, b): return a+b"]]
pass_at_k, _ = code_eval_metric.compute(references=test_cases, predictions=candidates, k=[1, 2])
print(pass_at_k)  # {'pass@1': 0.5, 'pass@2': 1.0}

## 1.6 LLM 기반 SE 접근법 — 4가지 패턴

1. **모델 선택 트레이드오프** (폐쇄형 성능 vs 오픈소스 비용·프라이버시)
2. **컨텍스트 윈도우 관리** (재귀 chunking + 계층 요약 → 최대 25% 개선)
3. **프레임워크 통합** (LangChain으로 보안 정책·피드백 루프)
4. **Human-AI Collaboration** (중요 결정은 사람, 반복은 AI)

> 핵심: 실행 피드백을 모델에 되먹여라.


## 1.7 보안 & 위험 완화 — LangChain 검증 체인

생성 코드는 표면적으로 정상이어도 취약점을 포함할 수 있다. SQL 인젝션·XSS·인증·민감정보 노출 등을 **Pydantic으로 정형 추출**해 자동 게이트키핑한다. (운영에서는 직접 실행 ✗, 샌드박스 필수)


In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_anthropic import ChatAnthropic


class SecurityAnalysis(BaseModel):
    vulnerabilities: list[str] = Field(..., description="발견된 취약점")
    mitigation_suggestions: list[str] = Field(..., description="완화 방안")
    risk_level: str = Field(..., description="Low | Medium | High | Critical")


parser = PydanticOutputParser(pydantic_object=SecurityAnalysis)
security_prompt = PromptTemplate(
    template="다음 코드의 보안을 분석하라.\n{format_instructions}\n\n코드:\n{code}\n",
    input_variables=["code"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)
security_chain = security_prompt | ChatAnthropic(model="claude-3-opus-20240229") | parser

sample = "import os\ndef run(cmd):\n    return os.system('ls ' + cmd)  # 사용자 입력 직접 사용"
result = security_chain.invoke({"code": sample})
print("위험도:", result.risk_level)
print("취약점:", result.vulnerabilities)

## 1.8 검증 프레임워크 — 운영 도입 전 6가지 점검

| 영역 | 점검 |
|------|------|
| Functional | 실행→출력 검증, 의존성, 요구사항 |
| Performance | 실행 시간, 스케일링, 메모리 |
| Security | 인젝션·인증·하드코딩 시크릿 |
| Robustness | 엣지 케이스, 잘못된 입력 |
| Business Logic | 도메인 규칙·규제, 수기 검증 |
| Documentation | 주석·가정·검증 보고서 |


## 1.9 LangChain 통합 — 도구 생태계

| 카테고리 | 도구 |
|----------|------|
| Code Execution | Python REPL · Azure Container Apps · Riza · Bearly |
| Database/Data | Cassandra · SQL · Spark SQL · JSON · pandas |
| Financial | FMP · Google Finance · FinancialDatasets · Dappier |
| Repo & VCS | GitHub · GitLab Toolkit |
| Input/Viz | Google Trends · PowerBI · Human-as-a-tool |


---
# Part 2 · Writing Code with LLMs
> Google · Hugging Face · Anthropic · Agent · RAG 비교


## 2.1 Google Gemini — FizzBuzz
LangChain 추상화 덕분에 모델만 교체하면 비교가 쉽다. 작은 함수 단위에서는 정확·깔끔한 풀이를 즉시 생성한다.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

question = """
Given an integer n, return a 1-indexed string array `answer` where:
- "FizzBuzz" if i % 15 == 0, "Fizz" if i % 3 == 0, "Buzz" if i % 5 == 0, else str(i).
Return Python code.
"""
llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro")
print(llm.invoke(question).content)

## 2.2 Hugging Face — 로컬 vs 추론 API
로컬은 통제·프라이버시 ↑(하드웨어 비용), Hub는 단순함·반응성 ↑.


In [ ]:
# (1) 로컬 파이프라인 — google/codegemma-2b (다운로드에 수 분 소요)
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline

checkpoint = "google/codegemma-2b"
pipe = pipeline(
    "text-generation",
    model=AutoModelForCausalLM.from_pretrained(checkpoint),
    tokenizer=AutoTokenizer.from_pretrained(checkpoint),
    max_new_tokens=500,
)
llm_local = HuggingFacePipeline(pipeline=pipe)

text = '''
def calculate_primes(n):
    """Return primes up to N.
    >>> calculate_primes(20)
    [2, 3, 5, 7, 11, 13, 17, 19]
    """
'''
print(llm_local.invoke(text))

In [ ]:
# (2) 추론 API — 다운로드 없이 호스팅 모델 호출
from huggingface_hub import InferenceClient

client = InferenceClient()
def complete(code, model="HuggingFaceH4/zephyr-7b-beta"):
    r = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": f"Continue this code:\n{code}\nLet's think step by step."}],
        max_tokens=128, temperature=0.5,
    )
    return r.choices[0].message.content

print(complete(text))

## 2.3 Anthropic Claude — 단계적 추론
코드와 설명을 함께 제공한다. 같은 템플릿에 **모델만 갈아 끼우면** 비교가 쉽다.


In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts.prompt import PromptTemplate

template = """Question: {question}
Let's think step by step.
Answer:
"""
prompt = PromptTemplate(template=template, input_variables=["question"])
chain = prompt | ChatAnthropic(model="claude-3-opus-20240229")
print(chain.invoke({"question": text}).content)

## 2.4 Agentic Approach — LLM이 코드를 쓰고 실행
`PythonREPLTool`로 ReAct 흐름(추론→실행→관찰→답변)을 수행한다.


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
from langchain_experimental.tools import PythonREPLTool

agent = initialize_agent(
    [PythonREPLTool()], ChatOpenAI(),
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True,
)
print(agent.invoke("What are the prime numbers until 20?"))

In [ ]:
# API 키 없이 흐름만 확인 — FakeListLLM
from langchain_core.language_models.fake import FakeListLLM
from langchain.agents import initialize_agent, AgentType
from langchain_experimental.tools import PythonREPLTool

responses = ["Action: Python_REPL\nAction Input: print(2 + 2)", "Final Answer: 4"]
fake = initialize_agent([PythonREPLTool()], FakeListLLM(responses=responses),
                        agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)
print(fake.invoke("What is 2 + 2?"))

## 2.5 Documentation RAG (참고 구조)
문서 로드→분할→임베딩→검색으로 인용 기반 답변. (DocusaurusLoader + Chroma + `rlm/rag-prompt`)


In [ ]:
# 개념 구조 — 실제 실행은 문서 로더/임베딩 설정 필요
# from langchain import hub
# rag_chain = (
#     {"context": retriever | format_docs, "question": RunnablePassthrough()}
#     | hub.pull("rlm/rag-prompt") | llm | StrOutputParser()
# )
# rag_chain.invoke("What is Task Decomposition?")

## 2.6 Repository RAG — 코드베이스 위 자연어 Q&A
언어 인식 파싱(LanguageParser) + MMR 검색. (한계: chunk_size=50은 맥락을 끊을 수 있음 → 크기 증가·계층 chunking 권장)


In [ ]:
import os
from git import Repo
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

repo_path = os.path.expanduser("~/Downloads/generative_ai_with_langchain")  # 아직 없어야 함
if not os.path.exists(repo_path):
    Repo.clone_from("https://github.com/benman1/generative_ai_with_langchain", to_path=repo_path)

documents = GenericLoader.from_filesystem(
    repo_path, glob="**/*", suffixes=[".py"],
    parser=LanguageParser(language=Language.PYTHON, parser_threshold=500),
).load()
texts = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=50, chunk_overlap=0
).split_documents(documents)
print(f"{len(documents)} docs -> {len(texts)} chunks")

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain

retriever = Chroma.from_documents(texts, OpenAIEmbeddings()).as_retriever(
    search_type="mmr", search_kwargs={"k": 8})
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer based on the context:\n\n{context}"),
    ("human", "{input}"),
])
qa = create_retrieval_chain(retriever, create_stuff_documents_chain(ChatOpenAI(), prompt))
print(qa.invoke({"input": "How is the agent memory implemented?"})["answer"])

---
# Part 3 · LLM Agents for Data Science
> 신경망 학습부터 데이터 탐색·시각화까지 — 자연어로


## 3.1 데이터 사이언스 + LLM — 보완의 관계

| LLM이 잘 하는 일 | 전통 통계·ML이 나은 일 |
|------------------|------------------------|
| 자연어↔코드(Text-to-SQL) | 정형 수치의 엄밀한 분석 |
| 전처리·시각화 코드 생성 | 복잡한 합성 데이터 생성 |
| 비기술 사용자 통역기 | 고위험 예측·의사결정 |
| 결과를 내러티브로 번역 | 재현성·해석 가능성 요구 |

> 복잡도가 증가할수록 LLM 코드 실행률 ↓ (PLOS One). 하이브리드가 가장 효과적.


## 3.2 Python 에이전트로 ML 학습 — 단일 뉴런 (y = 2x)
데이터 생성→모델 정의→학습→평가까지 **단일 프롬프트**로. (x=5 예측 ≈ 10)


In [ ]:
from langchain_experimental.agents.agent_toolkits.python.base import create_python_agent
from langchain_experimental.tools.python.tool import PythonREPLTool
from langchain_anthropic import ChatAnthropic
from langchain.agents import AgentType

agent_executor = create_python_agent(
    llm=ChatAnthropic(model="claude-3-opus-20240229"),
    tool=PythonREPLTool(), verbose=True,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
)
agent_executor.run("""
PyTorch로 단일 뉴런 신경망을 작성하라.
y = 2x 합성 데이터로 1000 epoch 학습, 100 epoch 마다 loss 출력.
마지막에 x=5 예측값을 출력하라.
""")

## 3.3 Pandas DataFrame 에이전트 — Iris
자연어로 탐색·계산·시각화. ⚠️ `allow_dangerous_code=True`는 임의 실행 허용 → 샌드박스 필수.


In [ ]:
import pandas as pd
from langchain_experimental.agents.agent_toolkits.pandas.base import create_pandas_dataframe_agent
from langchain_openai import ChatOpenAI

try:
    from sklearn.datasets import load_iris
    df = load_iris(as_frame=True).frame.rename(columns={
        "sepal length (cm)": "sepal_length", "sepal width (cm)": "sepal_width",
        "petal length (cm)": "petal_length", "petal width (cm)": "petal_width"})
except Exception:
    df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv")

agent = create_pandas_dataframe_agent(
    ChatOpenAI(model="gpt-4o-mini"), df, verbose=True, allow_dangerous_code=True)

print(agent.invoke("What's this dataset about?"))               # 탐색
print(agent.invoke("Which row has the biggest gap between petal length and width?"))  # 계산
agent.invoke("Show the distributions for each numeric column with matplotlib.")        # 시각화

## 3.4 대용량 데이터 다루기 — 3가지 전략
1. **요약 & 전처리** — shape·dtypes·요약통계·대표 샘플을 먼저 전달
2. **Chunking** — 세그먼트별 실행 후 결과 집계
3. **쿼리 특화 전처리** — 통계→사전집계, 상관→상관행렬, 탐색→메타데이터만


In [ ]:
# 전략 1 예시 — 에이전트에 넘기기 전 컨텍스트 요약
def summarize_for_agent(df, n=3):
    return {
        "shape": df.shape,
        "columns": list(df.columns),
        "dtypes": df.dtypes.astype(str).to_dict(),
        "describe": df.describe().round(2).to_dict(),
        "head": df.head(n).to_dict(orient="records"),
        "sample": df.sample(min(n, len(df)), random_state=0).to_dict(orient="records"),
    }

import json
print(json.dumps(summarize_for_agent(df), ensure_ascii=False, indent=2)[:800], "...")

## 3.5 데이터 사이언스 에이전트의 능력
1. **Code & Run** 2. **Train Models** 3. **Answer & Show** 4. **Automate**
> 탐색→전처리→분석→시각화→해석 전 영역을 커버하는 도구를 에이전트에게 제공하라.


---
# 정리 · Chapter 7 핵심

1. **자연어 = 새 인터페이스** — 코드를 대체하지 않고 '고수준 인터페이스'로 공존
2. **현실적 생산성** — 벤더 55%는 통제 결과, 실 환경 4–22%·88% 재작업 → 검증 필수
3. **보안 우선** — Pydantic 파서 + 샌드박스 + 의존성 검증 + 사람 리뷰의 다층 방어
4. **RAG로 맥락 확장** — 문서·코드 저장소 연결로 정확도 향상
5. **DS 에이전트** — Python·pandas 에이전트로 학습·분석·시각화. 단, 정밀 분석은 전통 기법이 강함

### 토론 질문
1. Vibe coding은 전통 코딩·유지보수를 어떻게 바꾸는가?
2. low-code vs LLM 자연어 개발의 차이는?
3. 벤더 주장과 독립 연구가 다른 이유는?
4. ClassEval이 시사하는 바는?
5. 검증 프레임워크 6영역의 중요성은?
6. Repository RAG를 수천 파일로 어떻게 확장할까?
7. 정형 vs 비정형 처리에서 LLM의 특성은?
8. 에이전트 기반 DS의 장점과 한계는?
9. 조직의 LLM 도구 도입 시 핵심 고려요소는?

> 다음 장 — Chapter 8. Evaluation and Testing.
